# YOLOv10m Training - Pothole Detection

This notebook demonstrates training YOLOv10m model for pothole detection.

## Features:
- Loads configuration from `.env` file
- Automatic model download
- Real-time training visualization
- Results saved to `runs/detect/pothole_detector/`

## 1. Setup & Configuration

First, let's set up the environment and load configuration from `.env`

In [ ]:
import sys
import os
import logging
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════════
# VERIFY PYTHON ENVIRONMENT - MUST USE venv-gpu WITH PYTHON 3.10.11
# ═══════════════════════════════════════════════════════════════════════════════

python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
python_path = sys.executable
venv_gpu_path = str(Path(__file__).parent.parent / 'venv-gpu').lower()

print("\n" + "="*70)
print("PYTHON ENVIRONMENT CHECK")
print("="*70)
print(f"Python Version:       {python_version}")
print(f"Python Executable:    {python_path}")
print(f"Expected Version:     3.10.11")
print("="*70)

# Check if using venv-gpu
if venv_gpu_path in python_path.lower():
    print("\n✓ Using venv-gpu environment")
else:
    print(f"\n⚠️  WARNING: Not using venv-gpu!")
    print(f"Expected path to contain: {venv_gpu_path}")
    print(f"Actual path: {python_path}")
    print("\nTo fix:")
    print("  1. Close this notebook")
    print("  2. In VS Code, click kernel selector (top right)")
    print("  3. Select: venv-gpu (or Python venv-gpu)")
    print("  4. Reopen the notebook")

# Check Python version
if python_version.startswith("3.10"):
    print(f"✓ Using Python 3.10.x (version {python_version})")
else:
    print(f"\n⚠️  WARNING: Python version mismatch!")
    print(f"Expected: 3.10.11")
    print(f"Got:      {python_version}")

print("="*70 + "\n")

# Disable font downloads to avoid issues
os.environ['YOLOV5_DISABLE_TELEMETRY'] = '1'

# Configure logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

print("✓ Environment configured!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFY .env FILE AND LOAD CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

backend_path = Path.cwd().parent / 'backend'
sys.path.insert(0, str(backend_path))

# Check if .env exists
project_root = Path.cwd().parent
env_file = project_root / '.env'

print("\n" + "="*70)
print(".env FILE CHECK")
print("="*70)

if env_file.exists():
    print(f"✓ .env file found at: {env_file}")
    print(f"  Size: {env_file.stat().st_size} bytes")
    
    # Show .env contents (without values for security)
    print(f"\n.env Configuration Keys:")
    with open(env_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                if '=' in line:
                    key = line.split('=')[0]
                    print(f"  ✓ {key}")
else:
    print(f"\n⚠️  WARNING: .env file not found!")
    print(f"  Expected location: {env_file}")
    print(f"\nTo fix:")
    print(f"  1. Copy .env.example to .env")
    print(f"  2. Edit .env and set your configuration")
    print(f"  3. Run this cell again")
    print(f"\nCreating from .env.example...")
    
    env_example = project_root / '.env.example'
    if env_example.exists():
        with open(env_example, 'r') as f:
            content = f.read()
        with open(env_file, 'w') as f:
            f.write(content)
        print(f"✓ .env created from .env.example")
    else:
        print(f"✗ .env.example not found either!")

print("="*70)

# Load configuration from .env
try:
    from config import config
    
    print("\n" + "="*70)
    print("CONFIGURATION LOADED FROM .env")
    print("="*70)
    print(f"Model Type:           {config.MODEL_TYPE}")
    print(f"Training Device:      {config.TRAIN_DEVICE}")
    print(f"Batch Size:           {config.TRAIN_BATCH_SIZE}")
    print(f"Epochs:               {config.TRAIN_EPOCHS}")
    print(f"Patience:             {config.TRAIN_PATIENCE}")
    print(f"Image Size:           {config.IMAGE_SIZE}")
    print(f"Number of Workers:    {config.NUM_WORKERS}")
    print("="*70)
    print("\n✓ Configuration loaded successfully!")
    print("Ready to start training!")
    
except Exception as e:
    print(f"\n✗ ERROR loading configuration: {e}")
    print(f"\nMake sure .env is properly configured!")
    raise


## 2. Import Required Libraries

Install and import ultralytics for YOLOv10

In [3]:
# Import ultralytics
try:
    from ultralytics import YOLO
    print("✓ Ultralytics imported successfully")
except ImportError:
    print("Installing ultralytics...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    from ultralytics import YOLO
    print("✓ Ultralytics installed and imported")

✓ Ultralytics installed and imported


## 3. Verify Dataset

Check that dataset and data.yaml are available

In [4]:
# Verify dataset paths
project_root = Path.cwd().parent
data_yaml = project_root / 'data' / 'data.yaml'
dataset_root = project_root / 'data'

print(f"Project root:        {project_root}")
print(f"Dataset folder:      {dataset_root}")
print(f"Data.yaml path:      {data_yaml}")

if data_yaml.exists():
    print("\n✓ data.yaml found!")
    with open(data_yaml, 'r') as f:
        print("\nDataset Configuration:")
        print(f.read())
else:
    print(f"\n✗ ERROR: data.yaml not found at {data_yaml}")
    print("Make sure you prepared the dataset first!")

Project root:        c:\Users\ihsan\Documents\GitHub\ML2
Dataset folder:      c:\Users\ihsan\Documents\GitHub\ML2\data
Data.yaml path:      c:\Users\ihsan\Documents\GitHub\ML2\data\data.yaml

✓ data.yaml found!

Dataset Configuration:
# YOLOv11 Dataset Configuration
# Pothole Detection Dataset

# Dataset paths
path: c:\Users\ihsan\Documents\GitHub\ML2\data
train: images/train
val: images/val

# Number of classes
nc: 2

# Class names
names:
  0: plain
  1: pothole



## 4. Load Model

Load the YOLOv10m pretrained model (will auto-download if not present)

In [5]:
print(f"\nLoading {config.MODEL_TYPE} model...")
print("(This may take a minute on first run - model will be downloaded)\n")

model = YOLO('yolov10m.pt')

print(f"✓ Model loaded successfully!")
print(f"Model type: {config.MODEL_TYPE}")
print(f"Model info:")
print(model.info())


Loading yolov10m model...
(This may take a minute on first run - model will be downloaded)

✓ Model loaded successfully!
Model type: yolov10m
Model info:
YOLOv10m summary: 288 layers, 16,576,768 parameters, 0 gradients, 64.5 GFLOPs
(288, 16576768, 0, 64.4772096)


## 5. Start Training

Begin training with configuration from .env file

In [6]:
print("\n" + "="*70)
print("YOLOv10m TRAINING - POTHOLE DETECTION")
print("="*70)
print(f"\nTraining Configuration:")
print(f"  Dataset:      {str(data_yaml)}")
print(f"  Epochs:       {config.TRAIN_EPOCHS}")
print(f"  Batch Size:   {config.TRAIN_BATCH_SIZE}")
print(f"  Image Size:   {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"  Device:       {config.TRAIN_DEVICE}")
print(f"  Patience:     {config.TRAIN_PATIENCE}")
print(f"  Workers:      {config.NUM_WORKERS}")

# Robust device check: treat numeric strings and 'cuda' as GPU
device_str = str(config.TRAIN_DEVICE).strip().lower()
if device_str in ('cpu', '', 'none', 'null'):
    print("\nRunning on CPU (slower)")
    print("To enable GPU, edit .env: TRAIN_DEVICE=0")
else:
    print(f"\nRunning on GPU (device {config.TRAIN_DEVICE})")
    # Try to print CUDA availability info
    try:
        import torch
        print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
        print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")
    except Exception as e:
        print(f"Could not import torch to check CUDA availability: {e}")

print("="*70)
print("Starting training next...\n")
print("="*70)


YOLOv10m TRAINING - POTHOLE DETECTION

Training Configuration:
  Dataset:      c:\Users\ihsan\Documents\GitHub\ML2\data\data.yaml
  Epochs:       5
  Batch Size:   4
  Image Size:   640x640
  Device:       0
  Patience:     10
  Workers:      2

Running on GPU (device 0)
torch.cuda.is_available(): True
torch.cuda.device_count(): 1
Starting training next...



In [7]:
# Validate parameters before training
import torch

print("\n" + "="*70)
print("VALIDATING TRAINING PARAMETERS")
print("="*70)
print(f"Data YAML exists: {data_yaml.exists()}")
print(f"Data YAML: {data_yaml}")
print(f"Device: {config.TRAIN_DEVICE} (type: {type(config.TRAIN_DEVICE).__name__})")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Epochs: {config.TRAIN_EPOCHS} (type: {type(config.TRAIN_EPOCHS).__name__})")
print(f"Batch size: {config.TRAIN_BATCH_SIZE} (type: {type(config.TRAIN_BATCH_SIZE).__name__})")
print(f"Image size: {config.IMAGE_SIZE} (type: {type(config.IMAGE_SIZE).__name__})")
print("="*70)

# Ensure device is valid
if not torch.cuda.is_available() and str(config.TRAIN_DEVICE).strip() not in ('cpu', '', '0'):
    print("\n⚠️  WARNING: CUDA not available, using CPU")
    device = 'cpu'
else:
    device = config.TRAIN_DEVICE

try:
    print("\n" + "="*70)
    print("STARTING TRAINING")
    print("="*70 + "\n")
    
    # Train the model
    results = model.train(
        data=str(data_yaml),
        epochs=config.TRAIN_EPOCHS,
        imgsz=config.IMAGE_SIZE,
        batch=config.TRAIN_BATCH_SIZE,
        device=device,
        patience=config.TRAIN_PATIENCE,
        save=True,
        project=str(project_root / 'runs' / 'detect'),
        name='pothole_detector',
        workers=config.NUM_WORKERS,
        close_mosaic=5,
        plots=True,
        verbose=True
    )
    
    print("\n" + "="*70)
    print("✓ TRAINING COMPLETE!")
    print("="*70)

except Exception as e:
    print("\n" + "="*70)
    print("✗ TRAINING ERROR")
    print("="*70)
    print(f"\nError type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    print("\nTroubleshooting tips:")
    print("  1. Check that TRAIN_DEVICE is set correctly in .env (use '0' for GPU or 'cpu')")
    print("  2. Verify data.yaml exists and paths are correct")
    print("  3. Ensure dataset has proper train/val folders with images and labels")
    print("  4. Check that TRAIN_EPOCHS and TRAIN_BATCH_SIZE are positive integers")
    print(f"\nFull traceback:")
    import traceback
    traceback.print_exc()


VALIDATING TRAINING PARAMETERS
Data YAML exists: True
Data YAML: c:\Users\ihsan\Documents\GitHub\ML2\data\data.yaml
Device: 0 (type: str)
CUDA available: True
Epochs: 5 (type: int)
Batch size: 4 (type: int)
Image size: 640 (type: int)

STARTING TRAINING

Ultralytics 8.4.7  Python-3.10.11 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\ihsan\Documents\GitHub\ML2\data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, l

## 6. Training Results

In [12]:
# Display training results - find the latest pothole_detector* run
from pathlib import Path
import os

detect_dir = project_root / 'runs' / 'detect'
pothole_runs = sorted([d for d in detect_dir.glob('pothole_detector*') if d.is_dir()], 
                      key=lambda x: os.path.getmtime(x), reverse=True)

if pothole_runs:
    results_dir = pothole_runs[0]  # Latest run
    print(f"Latest training run: {results_dir.name}")
else:
    results_dir = project_root / 'runs' / 'detect' / 'pothole_detector'
    print(f"Results directory: {results_dir}")

weights_dir = results_dir / 'weights'
best_model = weights_dir / 'best.pt'

print(f"\nResults Directory: {results_dir}")
print(f"\nOutput Files:")

if results_dir.exists():
    for file in sorted(results_dir.glob('*')):
        if file.is_file():
            size_mb = file.stat().st_size / (1024*1024)
            print(f"  - {file.name} ({size_mb:.2f} MB)")

print(f"\nWeights:")
if weights_dir.exists():
    for file in sorted(weights_dir.glob('*.pt')):
        size_mb = file.stat().st_size / (1024*1024)
        print(f"  ✓ {file.name} ({size_mb:.2f} MB)")

if best_model.exists():
    size_mb = best_model.stat().st_size / (1024*1024)
    print(f"\n✓ Best Model Found: {best_model}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"\n✗ Best model not found at {best_model}")

Latest training run: pothole_detector6

Results Directory: c:\Users\ihsan\Documents\GitHub\ML2\runs\detect\pothole_detector6

Output Files:
  - args.yaml (0.00 MB)
  - BoxF1_curve.png (0.08 MB)
  - BoxP_curve.png (0.08 MB)
  - BoxPR_curve.png (0.07 MB)
  - BoxR_curve.png (0.08 MB)
  - confusion_matrix.png (0.08 MB)
  - confusion_matrix_normalized.png (0.09 MB)
  - labels.jpg (0.08 MB)
  - results.csv (0.00 MB)
  - results.png (0.25 MB)
  - train_batch0.jpg (0.15 MB)
  - train_batch1.jpg (0.20 MB)
  - train_batch2.jpg (0.18 MB)
  - val_batch0_labels.jpg (0.43 MB)
  - val_batch0_pred.jpg (0.44 MB)

Weights:
  ✓ best.pt (31.92 MB)
  ✓ last.pt (31.92 MB)

✓ Best Model Found: c:\Users\ihsan\Documents\GitHub\ML2\runs\detect\pothole_detector6\weights\best.pt
  Size: 31.92 MB


## 7. Test Trained Model

Load the trained model and test on a sample image

In [ ]:
if best_model.exists():
    print(f"Loading trained model from {best_model}")
    trained_model = YOLO(str(best_model))
    print("✓ Trained model loaded successfully!")
    
    # Find a sample image to test
    data_folder = project_root / 'data' / 'images' / 'val'
    if data_folder.exists():
        images = list(data_folder.glob('*.jpg')) + list(data_folder.glob('*.png'))
        if images:
            test_image = images[0]
            print(f"\nTesting on: {test_image.name}")
            
            results = trained_model.predict(source=str(test_image), conf=0.5)
            print(f"✓ Prediction complete!")
            print(f"  Detections: {len(results[0].boxes)}")
        else:
            print("\nNo test images found in data/images/val/")
    else:
        print(f"\nValidation folder not found: {data_folder}")
else:
    print("\n✗ Trained model not found. Training may have failed.")

Loading trained model from c:\Users\ihsan\Documents\GitHub\ML2\runs\detect\pothole_detector6\weights\best.pt
✓ Trained model loaded successfully!

Testing on: 1.jpg

image 1/1 c:\Users\ihsan\Documents\GitHub\ML2\data\images\val\1.jpg: 480x640 1 pothole, 57.6ms
Speed: 13.4ms preprocess, 57.6ms inference, 3.7ms postprocess per image at shape (1, 3, 480, 640)
✓ Prediction complete!
  Detections: 1


: 

## 8. Summary

Training is complete! Here's what was accomplished:

In [10]:
print("\n" + "="*70)
print("TRAINING SUMMARY")
print("="*70)
print(f"\nModel Trained:        YOLOv10m")
print(f"Dataset:              Pothole Detection")
print(f"Training Epochs:      {config.TRAIN_EPOCHS}")
print(f"Device Used:          {config.TRAIN_DEVICE}")
print(f"Results Location:     {results_dir}")
print(f"Best Model:           {best_model}")

print(f"\nNext Steps:")
print(f"  1. Copy best.pt to backend/ for inference")
print(f"  2. Update .env: MODEL_PATH=path/to/best.pt")
print(f"  3. Run inference: python backend/app.py")
print(f"  4. Start frontend: cd frontend && npm run dev")

print(f"\nConfiguration for next run (in .env):")
print(f"  - Change MODEL_PATH to your trained model")
print(f"  - Adjust other settings as needed")
print(f"  - Run training again if needed")

print("\n" + "="*70)
print("Training notebook complete!")
print("="*70)


TRAINING SUMMARY

Model Trained:        YOLOv10m
Dataset:              Pothole Detection
Training Epochs:      5
Device Used:          0
Results Location:     c:\Users\ihsan\Documents\GitHub\ML2\runs\detect\pothole_detector
Best Model:           c:\Users\ihsan\Documents\GitHub\ML2\runs\detect\pothole_detector\weights\best.pt

Next Steps:
  1. Copy best.pt to backend/ for inference
  2. Update .env: MODEL_PATH=path/to/best.pt
  3. Run inference: python backend/app.py
  4. Start frontend: cd frontend && npm run dev

Configuration for next run (in .env):
  - Change MODEL_PATH to your trained model
  - Adjust other settings as needed
  - Run training again if needed

Training notebook complete!
